# ONNX INT8 Quantization — Decoder-only

**Chiến lược:** Chỉ quantize **text decoder** (INT8 dynamic), giữ **vision encoder FP32**.

Lý do:
- Vision encoder dùng Conv2d — rất nhạy cảm với quantization, mất accuracy nhiều
- Text decoder dùng Linear layers — INT8 dynamic weight-only an toàn hơn
- `reduce_range=True` tránh overflow trên CPU không có AVX-VNNI
- `MatMulConstBOnly=True` chỉ quantize weight cố định, giữ activation FP32

**Yêu cầu:** Phải chạy `onnx_export_fp32.ipynb` trước để có 3 file ONNX split.

In [1]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"ONNX Runtime: {ort.__version__}")

c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0530 22:39:09.880000 16072 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ONNX Runtime: 1.26.0


In [2]:
ROOT_DIR    = "../data_ready_for_kaggle"
TEST_PATH   = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR  = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR   = os.path.abspath("../saved_models_v3")
RESULTS_DIR = "./results"
SAMPLE_IMAGE = "../data_ready_for_kaggle/images_resized/00000006091.jpg"

# Input FP32 paths (split graphs từ onnx_export_fp32.ipynb)
FP32_DIR              = "./model_quantized/onnx_fp32"
ENCODER_FP32_PATH     = os.path.join(FP32_DIR, 'vision_encoder_fp32.onnx')
DEC_INIT_FP32_PATH    = os.path.join(FP32_DIR, 'text_decoder_init_fp32.onnx')
DEC_PAST_FP32_PATH    = os.path.join(FP32_DIR, 'text_decoder_with_past_fp32.onnx')

# Output INT8 paths (decoder quantized, encoder kept FP32)
INT8_DIR              = "./model_quantized/onnx_int8"
os.makedirs(INT8_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Vision encoder giữ nguyên FP32 — chỉ copy symlink logic, dùng lại file gốc
ENCODER_INT8_PATH     = ENCODER_FP32_PATH          # FP32, không quantize
DEC_INIT_INT8_PATH    = os.path.join(INT8_DIR, 'text_decoder_init_int8.onnx')
DEC_PAST_INT8_PATH    = os.path.join(INT8_DIR, 'text_decoder_with_past_int8.onnx')

MAX_TEST_IMAGES = 200
MAX_LENGTH      = 64
BATCH_SIZE      = 16
NUM_WORKERS     = 0
WARMUP_STEPS    = 3

# Kiểm tra file FP32 split có tồn tại không
for p in [ENCODER_FP32_PATH, DEC_INIT_FP32_PATH, DEC_PAST_FP32_PATH]:
    assert os.path.exists(p), f"File không tìm thấy: {p}\n→ Chạy onnx_export_fp32.ipynb trước!"
    size_mb = os.path.getsize(p) / 1e6
    print(f"OK  {os.path.basename(p):45s}  {size_mb:7.1f} MB")

OK  vision_encoder_fp32.onnx                         344.5 MB
OK  text_decoder_init_fp32.onnx                      851.5 MB
OK  text_decoder_with_past_fp32.onnx                 794.8 MB


## 1. Load model config (để biết num_layers, token IDs)

In [3]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

bos_token_id = model.config.text_config.bos_token_id
eos_token_id = model.config.text_config.eos_token_id
pad_token_id = model.config.text_config.pad_token_id
if pad_token_id is None:
    pad_token_id = eos_token_id

num_layers      = model.text_decoder.config.num_hidden_layers
patch_size      = model.vision_model.config.patch_size
image_size      = model.vision_model.config.image_size
encoder_seq_len = (image_size // patch_size) ** 2 + 1

print(f"num_layers     : {num_layers}")
print(f"encoder_seq_len: {encoder_seq_len}")

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2015.10it/s]

num_layers     : 12
encoder_seq_len: 577


## 2. Quantize Text Decoder — INT8 Dynamic (weight-only, decoder only)

- `weight_type=QInt8` — quantize weights thành INT8
- `reduce_range=True` — dùng 7-bit range (127 thay vì 255) tránh overflow Intel CPU
- `extra_options={'MatMulConstBOnly': True}` — chỉ quantize weight tensor cố định, activation giữ FP32
- Vision encoder **KHÔNG** quantize vì Conv2d rất nhạy với quantization

In [4]:
QUANT_CONFIG = dict(
    weight_type   = QuantType.QInt8,
    optimize_model= True,
    reduce_range  = True,            # safe cho Intel CPU không có VNNI
    extra_options = {'MatMulConstBOnly': True},  # weight-only, activation FP32
)

print("Quantizing text_decoder_init...")
quantize_dynamic(DEC_INIT_FP32_PATH, DEC_INIT_INT8_PATH, **QUANT_CONFIG)
print(f"  Saved: {DEC_INIT_INT8_PATH}")

print("Quantizing text_decoder_with_past...")
quantize_dynamic(DEC_PAST_FP32_PATH, DEC_PAST_INT8_PATH, **QUANT_CONFIG)
print(f"  Saved: {DEC_PAST_INT8_PATH}")

print("\n[Vision encoder giữ FP32 — không quantize]")

# Report size reduction
print("\n--- Size comparison (Text Decoder) ---")
for fp, ip, label in [
    (DEC_INIT_FP32_PATH,  DEC_INIT_INT8_PATH,  'decoder_init'),
    (DEC_PAST_FP32_PATH,  DEC_PAST_INT8_PATH,  'decoder_with_past'),
]:
    fp_mb = os.path.getsize(fp) / 1e6
    ip_mb = os.path.getsize(ip) / 1e6
    ratio = fp_mb / ip_mb
    print(f"  {label:25s}  FP32={fp_mb:.1f}MB  INT8={ip_mb:.1f}MB  ratio={ratio:.2f}x")

Quantizing text_decoder_init...


TypeError: quantize_dynamic() got an unexpected keyword argument 'optimize_model'

## 3. Benchmark ONNX INT8 — Vision FP32 + Decoder INT8

In [ ]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        df = pd.read_csv(data_file)
        self.img_dir    = img_dir
        self.processor  = processor
        self.max_length = max_length
        self.image_groups = (
            df.groupby('image_file')['caption'].apply(list).reset_index()
        )

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, idx):
        row        = self.image_groups.iloc[idx]
        image_file = row['image_file']
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')
        enc = self.processor(
            images=image, return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc['image_file']   = image_file
        enc['raw_captions'] = row['caption']
        return enc


def collate_fn(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ['pixel_values']}
    out['image_file']   = [b['image_file']   for b in batch]
    out['raw_captions'] = [b['raw_captions'] for b in batch]
    return out


def build_references(dataloader):
    refs = {}
    for batch in dataloader:
        for img, caps in zip(batch['image_file'], batch['raw_captions']):
            refs[img] = caps
    return refs


def calculate_metrics(preds_dict, refs_dict):
    common = sorted(set(preds_dict) & set(refs_dict))
    preds  = [preds_dict[k] for k in common]
    refs   = [refs_dict[k]  for k in common]
    bleu   = evaluate.load('bleu')
    rouge  = evaluate.load('rouge')
    meteor = evaluate.load('meteor')
    return {
        'bleu4':  bleu.compute(predictions=preds, references=refs)['bleu'] * 100,
        'rougeL': rouge.compute(predictions=preds, references=refs)['rougeL'] * 100,
        'meteor': meteor.compute(predictions=preds, references=refs)['meteor'] * 100,
        'num_images': len(common),
    }


def select_provider():
    avail = ort.get_available_providers()
    if 'CUDAExecutionProvider' in avail:
        return ['CUDAExecutionProvider', 'CPUExecutionProvider'], 'cuda', 'CUDAExecutionProvider'
    return ['CPUExecutionProvider'], 'cpu', 'CPUExecutionProvider'


def greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess,
                        pixel_values_np, max_length=64):
    """
    Greedy decode voi KV-cache.
    Inputs/Outputs kiem tra thuc te tu ONNX graph:
      decoder_init inputs : input_ids, attention_mask, encoder_hidden_states
                            (encoder_attention_mask bi constant-fold, khong truyen)
      decoder_past inputs : input_ids, attention_mask, past_self/cross_key/val_i
                            (encoder_hidden_states khong can, da bake vao cross-KV)
    QUAN TRONG: ham khong nhan processor lam tham so.
                processor la bien global trong notebook.
    """
    B = pixel_values_np.shape[0]

    # 1. Vision encode
    enc_hidden = enc_sess.run(None, {'pixel_values': pixel_values_np})[0]

    # 2. Decoder init (BOS token, khong co past KV)
    generated = np.full((B, 1), bos_token_id, dtype=np.int64)
    finished  = np.zeros(B, dtype=bool)

    init_outs = dec_init_sess.run(None, {
        'input_ids':             generated,
        'attention_mask':        np.ones_like(generated, dtype=np.int64),
        'encoder_hidden_states': enc_hidden,
    })
    logits = init_outs[0]  # (B, 1, vocab)

    # Map output pres_* -> past dict cho buoc tiep theo
    past = {}
    for i in range(num_layers):
        past[f'past_self_key_{i}']  = init_outs[1 + i*4]
        past[f'past_self_val_{i}']  = init_outs[2 + i*4]
        past[f'past_cross_key_{i}'] = init_outs[3 + i*4]
        past[f'past_cross_val_{i}'] = init_outs[4 + i*4]

    next_tok  = logits[:, -1, :].argmax(-1).astype(np.int64)
    next_tok  = np.where(finished, pad_token_id, next_tok)
    generated = np.concatenate([generated, next_tok[:, None]], axis=1)
    finished  = finished | (next_tok == eos_token_id)

    # 3. Step decode voi past KV
    for _ in range(max_length - 2):
        if finished.all():
            break
        dec_in = {
            'input_ids':      next_tok[:, None],
            'attention_mask': np.ones((B, generated.shape[1]), dtype=np.int64),
        }
        dec_in.update(past)
        step_outs = dec_past_sess.run(None, dec_in)
        logits    = step_outs[0]
        for i in range(num_layers):
            past[f'past_self_key_{i}']  = step_outs[1 + i*4]
            past[f'past_self_val_{i}']  = step_outs[2 + i*4]
            past[f'past_cross_key_{i}'] = step_outs[3 + i*4]
            past[f'past_cross_val_{i}'] = step_outs[4 + i*4]
        next_tok  = logits[:, -1, :].argmax(-1).astype(np.int64)
        next_tok  = np.where(finished, pad_token_id, next_tok)
        generated = np.concatenate([generated, next_tok[:, None]], axis=1)
        finished  = finished | (next_tok == eos_token_id)

    return processor.batch_decode(generated, skip_special_tokens=True)


print('Utilities defined.')


In [ ]:
provider_list, bench_device, provider_name = select_provider()
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# Vision encoder: FP32 session
enc_sess      = ort.InferenceSession(ENCODER_INT8_PATH,  sess_opts, providers=provider_list)
# Text decoders: INT8 sessions
dec_init_sess = ort.InferenceSession(DEC_INIT_INT8_PATH, sess_opts, providers=provider_list)
dec_past_sess = ort.InferenceSession(DEC_PAST_INT8_PATH, sess_opts, providers=provider_list)

print(f'Provider: {provider_name}  |  Device: {bench_device}')
print(f'Vision encoder : FP32  ({os.path.basename(ENCODER_INT8_PATH)})')
print(f'Decoder init   : INT8  ({os.path.basename(DEC_INIT_INT8_PATH)})')
print(f'Decoder past   : INT8  ({os.path.basename(DEC_PAST_INT8_PATH)})')


In [ ]:
test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

refs_dict = build_references(test_loader)
print(f'Test images: {len(test_dataset)}  |  batches: {len(test_loader)}')

# Warmup — chi truyen pixel_values_np (khong truyen processor)
sample_px = processor(
    images=Image.open(SAMPLE_IMAGE).convert('RGB'), return_tensors='pt'
)['pixel_values'].numpy().astype(np.float32)
_ = greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess, sample_px)
print('Warmup done.')


In [ ]:
preds_dict = {}
latencies  = []

for batch_idx, batch in enumerate(tqdm(test_loader, desc='Benchmarking ONNX INT8')):
    px = batch['pixel_values'].numpy().astype(np.float32)

    t0    = time.perf_counter()
    texts = greedy_decode_onnx(enc_sess, dec_init_sess, dec_past_sess, px, MAX_LENGTH)
    t1    = time.perf_counter()

    if batch_idx >= WARMUP_STEPS:
        latencies.append(t1 - t0)

    for img_f, text in zip(batch['image_file'], texts):
        preds_dict[img_f] = text.strip()

metrics    = calculate_metrics(preds_dict, refs_dict)
avg_lat    = float(np.mean(latencies))    if latencies else 0.0
p95_lat    = float(np.percentile(latencies, 95)) if latencies else 0.0
throughput = float(BATCH_SIZE / avg_lat)  if avg_lat > 0 else 0.0

benchmark = {
    'name':                           'onnx_int8',
    'backend':                        'onnx',
    'precision':                      'int8',
    'device':                         bench_device,
    'provider':                       provider_name,
    'generation_strategy':            'greedy+kvcache',
    'cache_enabled':                  True,
    'quantization_scope':             'decoder_only',
    'reduce_range':                   True,
    'batch_size':                     BATCH_SIZE,
    'max_test_images':                MAX_TEST_IMAGES,
    'avg_latency_seconds_per_batch':  avg_lat,
    'p95_latency_seconds_per_batch':  p95_lat,
    'throughput_images_per_second':   throughput,
    'bleu4':                          metrics['bleu4'],
    'rougeL':                         metrics['rougeL'],
    'meteor':                         metrics['meteor'],
    'num_images':                     metrics['num_images'],
    'model_paths': {
        'vision_encoder':    ENCODER_INT8_PATH,
        'decoder_init':      DEC_INIT_INT8_PATH,
        'decoder_with_past': DEC_PAST_INT8_PATH,
    },
}

print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'onnx_int8.json'), 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)
print("\nSaved: results/onnx_int8.json")